# # Chicago Taxi Trips — week of 11–17 September 2023
# 
# SUMMARY_PLACEHOLDER

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
FIG = Path("figures/analysis")
FIG.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv("trips.csv", parse_dates=["trip_start_timestamp", "trip_end_timestamp"])
print("shape:", raw.shape)
print(raw.dtypes)

shape: (135768, 23)
trip_id                                  str
taxi_id                                  str
trip_start_timestamp          datetime64[us]
trip_end_timestamp            datetime64[us]
trip_seconds                         float64
trip_miles                           float64
pickup_census_tract                  float64
dropoff_census_tract                 float64
pickup_community_area                float64
dropoff_community_area               float64
fare                                 float64
tips                                 float64
tolls                                float64
extras                               float64
trip_total                           float64
payment_type                             str
company                                  str
pickup_centroid_latitude             float64
pickup_centroid_longitude            float64
pickup_centroid_location                 str
dropoff_centroid_latitude            float64
dropoff_centroid_longitude         

In [2]:
print("Missing values per column:")
print(raw.isna().sum().to_frame("n_missing").assign(pct=lambda d: (d.n_missing / len(raw) * 100).round(2)))
print()
print("trip_id unique:", raw.trip_id.is_unique, "| duplicated full rows:", raw.duplicated().sum())
print("start range:", raw.trip_start_timestamp.min(), "->", raw.trip_start_timestamp.max())
print("end range:  ", raw.trip_end_timestamp.min(), "->", raw.trip_end_timestamp.max())

Missing values per column:
                            n_missing    pct
trip_id                             0   0.00
taxi_id                             0   0.00
trip_start_timestamp                0   0.00
trip_end_timestamp                  1   0.00
trip_seconds                       33   0.02
trip_miles                          0   0.00
pickup_census_tract             71404  52.59
dropoff_census_tract            73842  54.39
pickup_community_area            3815   2.81
dropoff_community_area          13794  10.16
fare                              324   0.24
tips                              324   0.24
tolls                             324   0.24
extras                            324   0.24
trip_total                        324   0.24
payment_type                        0   0.00
company                             0   0.00
pickup_centroid_latitude         3775   2.78
pickup_centroid_longitude        3775   2.78
pickup_centroid_location         3775   2.78
dropoff_centroid_latitude   

In [3]:
num = ["trip_seconds", "trip_miles", "fare", "tips", "tolls", "extras", "trip_total"]
print(raw[num].describe(percentiles=[.01, .05, .5, .95, .99, .999]).round(2).T)
print()
print(raw.payment_type.value_counts(dropna=False))
print()
print(raw.company.value_counts(dropna=False).head(15))

                 count     mean      std  min    1%     5%      50%      95%      99%     99.9%       max
trip_seconds  135735.0  1346.54  1762.25  0.0  0.00  22.00  1020.00  3382.30  4872.98  12416.09  86160.00
trip_miles    135768.0     6.94     7.85  0.0  0.00   0.00     3.24    18.76    27.14     44.08    792.90
fare          135444.0    23.87    19.60  0.0  3.25   4.25    17.75    53.50    75.50    150.00    911.00
tips          135444.0     3.38     4.67  0.0  0.00   0.00     2.00    11.88    17.10     33.33    130.00
tolls         135444.0     0.13    20.42  0.0  0.00   0.00     0.00     0.00     0.00      5.00   4444.44
extras        135444.0     2.54    18.31  0.0  0.00   0.00     0.00     7.00    35.50     65.75   4445.44
trip_total    135444.0    30.12    43.93  0.0  3.25   5.25    21.25    70.50   104.43    186.53   8901.13

payment_type
Credit Card    60291
Cash           40053
Mobile         18594
Prcard         11961
Unknown         4784
No Charge         54
Dispute     

In [4]:
print(raw.company.value_counts().to_string())
print("\nn companies:", raw.company.nunique())

company
Flash Cab                               25950
Taxi Affiliation Services               25727
Sun Taxi                                15122
Taxicab Insurance Agency Llc            13914
City Service                            13747
Chicago Independents                     8004
5 Star Taxi                              6121
Medallion Leasin                         5003
Globe Taxi                               4772
Blue Ribbon Taxi Association             4390
Star North Taxi Management Llc           2815
Taxicab Insurance Agency, LLC            2379
Choice Taxi Association                  2022
Chicago City Taxi Association            1386
Choice Taxi Association Inc              1064
U Taxicab                                1048
Chicago Taxicab                           406
Patriot Taxi Dba Peace Taxi Associat      395
Top Cab                                   364
Koam Taxi Association                     350
312 Medallion Management Corp             273
Setare Inc                

In [5]:
# Money rows: are the 324 missing-money rows the same rows? what do they look like?
m = raw[raw.fare.isna()]
print("rows with missing fare:", len(m), "| of which also missing trip_total:", m.trip_total.isna().sum())
print(m[["trip_seconds","trip_miles","payment_type","company"]].describe(include="all").T)
print(m.payment_type.value_counts())

rows with missing fare: 324 | of which also missing trip_total: 324
              count unique                           top freq        mean          std  min     25%    50%      75%     max
trip_seconds  316.0    NaN                           NaN  NaN  1231.85443  1166.560076  2.0  398.75  727.0   1923.5  7278.0
trip_miles    324.0    NaN                           NaN  NaN    7.143765     8.608393  0.0  0.8775   2.32  12.9125   34.65
payment_type    324      2                          Cash  323         NaN          NaN  NaN     NaN    NaN      NaN     NaN
company         324     12  Taxicab Insurance Agency Llc  104         NaN          NaN  NaN     NaN    NaN      NaN     NaN
payment_type
Cash           323
Credit Card      1
Name: count, dtype: int64


In [6]:
# trip_total vs sum of parts
parts = raw[["fare","tips","tolls","extras"]].sum(axis=1, min_count=4)
diff = (raw.trip_total - parts)
print("trip_total - sum(parts): non-null", diff.notna().sum())
print(diff.describe(percentiles=[.001,.01,.5,.99,.999]).round(3))
print("abs diff > 0.005:", (diff.abs() > 0.005).sum(), "| > 0.5:", (diff.abs() > 0.5).sum(), "| > 5:", (diff.abs() > 5).sum())
print(raw.loc[diff.abs() > 0.5, ["fare","tips","tolls","extras","trip_total","trip_miles","trip_seconds","payment_type"]].assign(diff=diff).sort_values("diff").head(15))
print(raw.loc[diff.abs() > 0.5, ["fare","tips","tolls","extras","trip_total"]].assign(diff=diff).sort_values("diff").tail(10))

trip_total - sum(parts): non-null 135444
count    135444.000
mean          0.204
std           0.365
min          -0.000
0.1%         -0.000
1%            0.000
50%           0.000
99%           2.500
99.9%         3.000
max           3.000
dtype: float64
abs diff > 0.005: 48197 | > 0.5: 1961 | > 5: 0
        fare  tips  tolls  extras  trip_total  trip_miles  trip_seconds payment_type  diff
23789   5.75  0.94    0.0     0.0        7.19        0.74         353.0       Mobile   0.5
50011   5.75  1.88    0.0     0.0        8.13        0.46         413.0       Mobile   0.5
69562   5.75  0.94    0.0     0.0        7.19        0.54         354.0       Mobile   0.5
110285  4.75  1.88    0.0     1.0        8.13        0.53         203.0       Mobile   0.5
5416    6.50  1.05    0.0     0.0        8.05        1.06         435.0       Mobile   0.5
18238   6.50  1.05    0.0     0.0        8.05        0.87         495.0       Mobile   0.5
91437   5.75  0.94    0.0     0.0        7.19        0.49   

In [7]:
d = diff.dropna().round(2)
print("Distribution of gap (trip_total - sum of parts), top values:")
print(d.value_counts().head(10))
print()
print("Share of trips with a gap > $0.005, by payment type:")
print((diff.abs() > 0.005).groupby(raw.payment_type).mean().round(3))
print()
print("Gap as % of trip_total (trips with gap): median", (diff / raw.trip_total)[diff > 0.005].median().round(4),
      "| total $ gap:", diff.sum().round(0), "of", raw.trip_total.sum().round(0))

Distribution of gap (trip_total - sum of parts), top values:
0.00    87247
0.50    46522
3.00      804
2.50      751
1.00      109
1.95        5
1.50        4
2.00        2
Name: count, dtype: int64

Share of trips with a gap > $0.005, by payment type:
payment_type
Cash           0.017
Credit Card    0.719
Dispute        0.000
Mobile         0.224
No Charge      0.000
Prcard         0.001
Unknown        0.000
dtype: float64

Gap as % of trip_total (trips with gap): median 0.0119 | total $ gap: 27679.0 of 4079349.0


In [8]:
r = raw
print("trip_seconds == 0:", (r.trip_seconds == 0).sum(), "| missing:", r.trip_seconds.isna().sum(), "| < 60s:", (r.trip_seconds < 60).sum())
print("trip_miles == 0:", (r.trip_miles == 0).sum(), "| > 100:", (r.trip_miles > 100).sum(), "| > 60:", (r.trip_miles > 60).sum())
print("fare == 0:", (r.fare == 0).sum(), "| trip_total == 0:", (r.trip_total == 0).sum(), "| fare > 200:", (r.fare > 200).sum(), "| trip_total > 300:", (r.trip_total > 300).sum())
print("trip_seconds > 6h:", (r.trip_seconds > 6*3600).sum(), "| > 12h:", (r.trip_seconds > 12*3600).sum())
print("tolls > 20:", (r.tolls > 20).sum(), "| extras > 100:", (r.extras > 100).sum())
print("end < start:", (r.trip_end_timestamp < r.trip_start_timestamp).sum(), "| end - start > 1 day:", ((r.trip_end_timestamp - r.trip_start_timestamp) > pd.Timedelta("1D")).sum())
print()
print("Extreme rows:")
cols = ["trip_seconds","trip_miles","fare","tips","tolls","extras","trip_total","payment_type","company"]
print(r.sort_values("trip_total", ascending=False)[cols].head(8))
print(r.sort_values("trip_miles", ascending=False)[cols].head(8))
print(r.sort_values("trip_seconds", ascending=False)[cols].head(8))

trip_seconds == 0: 2731 | missing: 33 | < 60s: 7963
trip_miles == 0: 13993 | > 100: 11 | > 60: 45
fare == 0: 135 | trip_total == 0: 134 | fare > 200: 49 | trip_total > 300: 38
trip_seconds > 6h: 98 | > 12h: 61
tolls > 20: 54 | extras > 100: 15
end < start: 0 | end - start > 1 day: 24

Extreme rows:
        trip_seconds  trip_miles    fare  tips    tolls   extras  trip_total payment_type                        company
124235        1080.0        0.10   11.25   0.0  4444.44  4445.44     8901.13         Cash      Taxi Affiliation Services
25636            0.0        0.00    3.25   0.0  4444.44  4444.44     8892.13         Cash      Taxi Affiliation Services
35524          420.0        1.30    6.75   0.0  4111.10     0.44     4118.29         Cash      Taxi Affiliation Services
127990           NaN       37.45  911.00   0.0     0.00     0.00      911.00         Cash   Blue Ribbon Taxi Association
31648            3.0        0.00  900.00   0.0     0.00     0.00      900.00         Cash      

In [9]:
# Implied speed and fare-per-mile sanity, for trips with positive miles & seconds
v = r[(r.trip_seconds > 0) & (r.trip_miles > 0)]
mph = v.trip_miles / (v.trip_seconds / 3600)
print("implied mph percentiles:\n", mph.describe(percentiles=[.5,.9,.99,.999]).round(1))
print("trips > 80 mph:", (mph > 80).sum())
print()
print("miles == 0 but seconds > 0 and fare > 10:", ((r.trip_miles == 0) & (r.trip_seconds > 0) & (r.fare > 10)).sum())
print("seconds == 0 or NaN but miles > 1:", (((r.trip_seconds == 0) | r.trip_seconds.isna()) & (r.trip_miles > 1)).sum())
print("payment types among fare==0:", r[r.fare == 0].payment_type.value_counts().to_dict())

implied mph percentiles:
 count    121660.0
mean         19.9
std         212.2
min           0.0
50%          15.0
90%          34.8
99%          50.9
99.9%        87.0
max       61992.0
dtype: float64
trips > 80 mph: 134

miles == 0 but seconds > 0 and fare > 10: 5212


seconds == 0 or NaN but miles > 1: 38
payment types among fare==0: {'Cash': 131, 'Credit Card': 2, 'No Charge': 1, 'Unknown': 1}
